# 10B — H3 Mutual Information Feature Selection

## Purpose
The original Script 10 performed worse than Script 09. This notebook tests one focused hypothesis: **are noisy/irrelevant features hurting H3?**

Mutual Information (MI) ranking is calculated **inside each training fold only**. The same Top-K grid and model search are applied to USD, GBP, EUR and JPY. The 2024–2025 holdout is never loaded.

For each outer year Y:
1. Train/feature-rank using data before Y-1.
2. Select Top-K/model/window/shrinkage on Y-1 only.
3. Freeze that configuration.
4. Evaluate on Y.

Script 09 remains the incumbent and is only replaced for a year when the MI model had lower RMSE on the previous-year inner validation.


In [ ]:
from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import t as student_t
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, HuberRegressor, ElasticNet
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 230)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES=["USD","GBP","EUR","JPY"]
H=3
OUTER_YEARS=[2019,2020,2021,2022,2023]
FEATURE_FAMILIES=["TT_OWN","TT_CROSS","FULL_PIT"]
TOP_K_GRID=[10,20,30,40,60]
TRAIN_WINDOWS=["RECENT_3Y","RECENT_5Y","EXPANDING"]
MODEL_KEYS=["RIDGE_1","RIDGE_10","HUBER","ELASTIC_NET","EXTRA_TREES","HIST_GB"]
SHRINKAGE_GRID=[0.0,0.1,0.2,0.3,0.4,0.5,0.75,1.0]
RANDOM_STATE=42


In [ ]:
def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        dev=root/"data"/"processed"/"02_development_model_dataset.csv"
        feat=root/"data"/"processed"/"02_feature_sets.json"
        rw=root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        on9=root/"outputs"/"predictions"/"09_online_hybrid_predictions_2019_2023.csv"
        if dev.exists() and feat.exists() and rw.exists() and on9.exists(): return root.resolve()
    raise FileNotFoundError("Required development files not found.")

ROOT=find_root()
DEV_PATH=ROOT/"data"/"processed"/"02_development_model_dataset.csv"
FEATURE_PATH=ROOT/"data"/"processed"/"02_feature_sets.json"
PRED_DIR=ROOT/"outputs"/"predictions"
TABLE_DIR=ROOT/"outputs"/"tables"
TABLE_DIR.mkdir(parents=True,exist_ok=True)
RW_PATH=PRED_DIR/"04_random_walk_predictions_2019_2023.csv"
ONLINE09_PATH=PRED_DIR/"09_online_hybrid_predictions_2019_2023.csv"

df=pd.read_csv(DEV_PATH,parse_dates=["Date"])
df["Target_Date_H3"]=pd.to_datetime(df["Target_Date_H3"],errors="coerce")
with open(FEATURE_PATH,"r",encoding="utf-8") as f: FEATURE_SETS=json.load(f)
assert df["Date"].max()<pd.Timestamp("2024-01-01")
print("PASS — 2024–2025 holdout remains untouched.")


In [ ]:
rw=pd.read_csv(RW_PATH,parse_dates=["Forecast_Origin","Target_Date"])
on9=pd.read_csv(ONLINE09_PATH,parse_dates=["Forecast_Origin","Target_Date"])
rw=rw[rw["Horizon"]==3].copy(); on9=on9[on9["Horizon"]==3].copy()
base09=rw[["Currency","Horizon","Forecast_Origin","Target_Date","Actual","RW_Prediction"]].merge(
    on9[["Currency","Horizon","Forecast_Origin","Online_Hybrid_Prediction"]],
    on=["Currency","Horizon","Forecast_Origin"],how="inner")
base09["Validation_Year"]=base09["Forecast_Origin"].dt.year
print("Matched Script 09 H3 rows:",len(base09))


In [ ]:
def rmse(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.sqrt(np.mean((y[m]-p[m])**2))) if m.sum() else np.nan
def mae(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float); m=np.isfinite(y)&np.isfinite(p)
    return float(np.mean(np.abs(y[m]-p[m]))) if m.sum() else np.nan
def skill(m,r): return float(1-m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def theils_u(m,r): return float(m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def dm_test(y,model,rw_pred,horizon=3):
    y=np.asarray(y,float); model=np.asarray(model,float); rw_pred=np.asarray(rw_pred,float)
    mask=np.isfinite(y)&np.isfinite(model)&np.isfinite(rw_pred); y=y[mask]; model=model[mask]; rw_pred=rw_pred[mask]
    n=len(y)
    if n<20: return np.nan,np.nan
    d=(y-rw_pred)**2-(y-model)**2; lag=max(0,horizon-1); lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if len(d)<=k: break
        lrv += 2*(1-k/(lag+1))*np.cov(d[k:],d[:-k],ddof=1)[0,1]
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n); p=2*student_t.sf(abs(stat),df=n-1)
    return float(stat),float(p)


## Candidate features
We keep the existing point-in-time feature families and add compact H3 technical features. MI then ranks the available features using the training sample only.

In [ ]:
for cur in CURRENCIES:
    mid=df[f"{cur}_mid"]; r=np.log(mid/mid.shift(1))
    df[f"{cur}_MI_ret1"]=r
    for w in [2,3,5,7]: df[f"{cur}_MI_ret{w}sum"]=r.rolling(w).sum()
    for w in [3,5,10,20]:
        df[f"{cur}_MI_vol{w}"]=r.rolling(w).std()
        ma=mid.rolling(w).mean(); df[f"{cur}_MI_ma{w}gap"]=(mid-ma)/mid

def candidate_features(cur,family):
    feats=[c for c in FEATURE_SETS[cur][family] if c in df.columns]
    engineered=[]
    for c in CURRENCIES:
        engineered += [f"{c}_MI_ret1",f"{c}_MI_ret3sum",f"{c}_MI_ret5sum",f"{c}_MI_ret7sum",
                       f"{c}_MI_vol5",f"{c}_MI_vol10",f"{c}_MI_ma5gap",f"{c}_MI_ma10gap"]
    return list(dict.fromkeys([c for c in feats+engineered if c in df.columns]))


In [ ]:
def mi_select(train,features,target,top_k):
    work=train[features+[target]].copy(); work=work[work[target].notna()]
    X=work[features]; y=work[target].to_numpy(float)
    imp=SimpleImputer(strategy="median"); X_imp=imp.fit_transform(X)
    scores=mutual_info_regression(X_imp,y,random_state=RANDOM_STATE)
    ranking=pd.DataFrame({"Feature":features,"MI_Score":scores}).sort_values("MI_Score",ascending=False).reset_index(drop=True)
    return ranking.head(min(top_k,len(ranking)))["Feature"].tolist(), ranking

def build_model(key):
    if key=="RIDGE_1": return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),("m",Ridge(alpha=1.0))])
    if key=="RIDGE_10": return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),("m",Ridge(alpha=10.0))])
    if key=="HUBER": return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),("m",HuberRegressor(epsilon=1.35,alpha=0.001,max_iter=500))])
    if key=="ELASTIC_NET": return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),("m",ElasticNet(alpha=0.001,l1_ratio=0.2,max_iter=5000,random_state=42))])
    if key=="EXTRA_TREES": return Pipeline([("i",SimpleImputer(strategy="median")),("m",ExtraTreesRegressor(n_estimators=140,max_depth=6,min_samples_leaf=10,max_features=.7,random_state=42,n_jobs=-1))])
    if key=="HIST_GB": return Pipeline([("i",SimpleImputer(strategy="median")),("m",HistGradientBoostingRegressor(learning_rate=.03,max_iter=150,max_leaf_nodes=15,l2_regularization=3.0,random_state=42))])
    raise ValueError(key)


In [ ]:
def apply_window(train,start,window):
    if window=="RECENT_3Y": return train[train["Date"]>=start-pd.DateOffset(years=3)].copy()
    if window=="RECENT_5Y": return train[train["Date"]>=start-pd.DateOffset(years=5)].copy()
    return train.copy()

def make_batch(cur,family,window,val_year):
    features=candidate_features(cur,family); target=f"{cur}_change_H3"
    start=pd.Timestamp(f"{val_year}-01-01"); end=pd.Timestamp(f"{val_year+1}-01-01")
    train=df[(df["Date"]<start)&df[target].notna()&df["Target_Date_H3"].notna()&(df["Target_Date_H3"]<start)].copy()
    train=apply_window(train,start,window)
    val=df[(df["Date"]>=start)&(df["Date"]<end)&df[target].notna()&df["Target_Date_H3"].notna()].copy()
    return train,val,features,target


In [ ]:
def tune_configuration(cur,outer_year):
    inner_year=outer_year-1; rows=[]
    for family in FEATURE_FAMILIES:
        for window in TRAIN_WINDOWS:
            train,val,features,target=make_batch(cur,family,window,inner_year)
            if len(train)<200 or len(val)<30 or not features: continue
            for top_k in TOP_K_GRID:
                try: selected,_=mi_select(train,features,target,top_k)
                except Exception: continue
                actual=val[f"{cur}_target_H3"].to_numpy(float); rwv=val[f"{cur}_mid"].to_numpy(float)
                for mk in MODEL_KEYS:
                    try:
                        m=build_model(mk); m.fit(train[selected],train[target]); raw=rwv+m.predict(val[selected])
                    except Exception: continue
                    for lam in SHRINKAGE_GRID:
                        pred=rwv+lam*(raw-rwv)
                        rows.append({"Feature_Family":family,"Train_Window":window,"Top_K":top_k,"Model":mk,"Lambda":lam,"Inner_RMSE":rmse(actual,pred)})
    tab=pd.DataFrame(rows)
    if tab.empty: return None,tab
    b=tab.sort_values(["Inner_RMSE","Top_K"]).iloc[0]
    return {"Feature_Family":b["Feature_Family"],"Train_Window":b["Train_Window"],"Top_K":int(b["Top_K"]),"Model":b["Model"],"Lambda":float(b["Lambda"]),"Inner_RMSE":float(b["Inner_RMSE"])},tab

def predict_outer(cur,year,sel):
    train,outer,features,target=make_batch(cur,sel["Feature_Family"],sel["Train_Window"],year)
    selected,ranking=mi_select(train,features,target,sel["Top_K"])
    exp=base09[(base09["Currency"]==cur)&(base09["Validation_Year"]==year)][["Forecast_Origin","Target_Date","Actual","RW_Prediction","Online_Hybrid_Prediction"]].copy()
    outer=outer.merge(exp,left_on="Date",right_on="Forecast_Origin",how="inner")
    m=build_model(sel["Model"]); m.fit(train[selected],train[target]); change=m.predict(outer[selected])
    rwv=outer["RW_Prediction"].to_numpy(float); mi_pred=rwv+sel["Lambda"]*change
    pred=pd.DataFrame({"Currency":cur,"Horizon":3,"Validation_Year":year,"Forecast_Origin":outer["Forecast_Origin"],"Target_Date":outer["Target_Date"],"Actual":outer["Actual"],"RW_Prediction":outer["RW_Prediction"],"Script09_Prediction":outer["Online_Hybrid_Prediction"],"MI_H3_Prediction":mi_pred})
    return pred,ranking


## Conservative incumbent rule
Script 09 is kept unless the MI model beats Script 09 on the previous inner-validation year. This prevents forcing MI into every period.

In [ ]:
all_pred=[]; sel_rows=[]; candidate_tabs=[]; rank_tabs=[]
for cur in CURRENCIES:
    print(f"\n===== {cur} H3 =====",flush=True)
    for year in OUTER_YEARS:
        t0=time.time(); sel,cands=tune_configuration(cur,year)
        if sel is None: continue
        pred,rank=predict_outer(cur,year,sel)
        inner_year=year-1
        inner09=base09[(base09["Currency"]==cur)&(base09["Validation_Year"]==inner_year)]
        s09_inner=rmse(inner09["Actual"],inner09["Online_Hybrid_Prediction"]) if len(inner09) else np.nan
        use_mi=(year!=2019 and np.isfinite(s09_inner) and sel["Inner_RMSE"]<s09_inner)
        pred["Use_MI"]=use_mi
        pred["Final_Prediction"]=np.where(use_mi,pred["MI_H3_Prediction"],pred["Script09_Prediction"])
        pred["Selected_Method"]=np.where(use_mi,"MI_H3","SCRIPT09")
        all_pred.append(pred)
        sel_rows.append({"Currency":cur,"Outer_Year":year,**sel,"Script09_Inner_RMSE":s09_inner,"Use_MI":use_mi})
        z=cands.copy(); z["Currency"]=cur; z["Outer_Year"]=year; candidate_tabs.append(z)
        r=rank.copy(); r["Currency"]=cur; r["Outer_Year"]=year; rank_tabs.append(r)
        print(f"{year} | TopK={sel['Top_K']} | {sel['Model']} | {sel['Feature_Family']} | {sel['Train_Window']} | lambda={sel['Lambda']:.2f} | use_MI={use_mi} | {time.time()-t0:.1f}s",flush=True)

final_predictions=pd.concat(all_pred,ignore_index=True)
selections=pd.DataFrame(sel_rows)


In [ ]:
pooled=[]
for cur,g in final_predictions.groupby("Currency"):
    rr=rmse(g["Actual"],g["RW_Prediction"]); r09=rmse(g["Actual"],g["Script09_Prediction"]); rmi=rmse(g["Actual"],g["MI_H3_Prediction"]); rf=rmse(g["Actual"],g["Final_Prediction"])
    _,p=dm_test(g["Actual"],g["Final_Prediction"],g["RW_Prediction"],3)
    pooled.append({"Currency":cur,"Horizon":3,"Observations":len(g),"RW_RMSE":rr,"Script09_RMSE":r09,"Raw_MI_RMSE":rmi,"Final_RMSE":rf,"Script09_Skill":skill(r09,rr),"Final_Skill":skill(rf,rr),"Final_Theils_U":theils_u(rf,rr),"DM_pvalue_vs_RW":p,"Beats_RW":rf<rr})
pooled_results=pd.DataFrame(pooled).sort_values("Currency")
display(pooled_results)
print("H3 tasks beating RW:",int(pooled_results["Beats_RW"].sum()),"/",len(pooled_results))


In [ ]:
yearly=[]
for (cur,year),g in final_predictions.groupby(["Currency","Validation_Year"]):
    rr=rmse(g["Actual"],g["RW_Prediction"]); mr=rmse(g["Actual"],g["Final_Prediction"]); _,p=dm_test(g["Actual"],g["Final_Prediction"],g["RW_Prediction"],3)
    yearly.append({"Currency":cur,"Horizon":3,"Validation_Year":year,"Observations":len(g),"RMSE":mr,"MAE":mae(g["Actual"],g["Final_Prediction"]),"Forecast_Skill_vs_RW":skill(mr,rr),"Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p,"Selected_Method":g["Selected_Method"].mode().iloc[0]})
yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Validation_Year"])
display(yearly_results)


In [ ]:
final_predictions.to_csv(PRED_DIR/"10B_H3_MI_predictions_2019_2023.csv",index=False)
pooled_results.to_csv(TABLE_DIR/"10B_H3_MI_pooled_results.csv",index=False)
yearly_results.to_csv(TABLE_DIR/"10B_H3_MI_yearly_results.csv",index=False)
selections.to_csv(TABLE_DIR/"10B_H3_MI_selections.csv",index=False)
pd.concat(candidate_tabs,ignore_index=True).to_csv(TABLE_DIR/"10B_H3_MI_candidates.csv",index=False)
pd.concat(rank_tabs,ignore_index=True).to_csv(TABLE_DIR/"10B_H3_MI_rankings.csv",index=False)

quality_gate=pd.DataFrame([
    {"Area":"Holdout isolation","Status":"PASS","Evidence":"2024–2025 not loaded"},
    {"Area":"MI leakage control","Status":"PASS","Evidence":"MI calculated inside each training fold only"},
    {"Area":"Common selection rule","Status":"PASS","Evidence":"Same Top-K/model/window/shrinkage search for all currencies"},
    {"Area":"Temporal tuning","Status":"PASS","Evidence":"Configuration selected on previous-year validation"},
    {"Area":"Incumbent protection","Status":"PASS","Evidence":"Script09 retained unless MI wins on prior inner validation"}
])
display(quality_gate)
print("\nScript 10B complete. 2024–2025 remains untouched.")
